In [ ]:
# !python main.py --config configs/config_mb_lenet.txt

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from e2cnn import gspaces
from e2cnn import nn as e2nn

import utils


class DNSteerableLeNet(nn.Module):
    def __init__(self, in_chan, out_chan, imsize, kernel_size=5, N=8):
        super(DNSteerableLeNet, self).__init__()
        
        z = 0.5*(imsize - 2)
        z = int(0.5*(z - 2))
        
        self.r2_act = gspaces.FlipRot2dOnR2(N)
        
        in_type = e2nn.FieldType(self.r2_act, [self.r2_act.trivial_repr])
        self.input_type = in_type
        
        out_type = e2nn.FieldType(self.r2_act, 6*[self.r2_act.regular_repr])
        self.mask = e2nn.MaskModule(in_type, imsize, margin=1)
        self.conv1 = e2nn.R2Conv(in_type, out_type, kernel_size=5, padding=1, bias=False)
        self.relu1 = e2nn.ReLU(out_type, inplace=True)
        self.pool1 = e2nn.PointwiseMaxPoolAntialiased(out_type, kernel_size=2)

        in_type = self.pool1.out_type
        out_type = e2nn.FieldType(self.r2_act, 16*[self.r2_act.regular_repr])
        self.conv2 = e2nn.R2Conv(in_type, out_type, kernel_size=5, padding=1, bias=False)
        self.relu2 = e2nn.ReLU(out_type, inplace=True)
        self.pool2 = e2nn.PointwiseMaxPoolAntialiased(out_type, kernel_size=2)
        
        self.gpool = e2nn.GroupPooling(out_type)

        self.fc1   = nn.Linear(16*z*z, 120)
        self.fc2   = nn.Linear(120, 84)
        self.fc3   = nn.Linear(84, out_chan)
        
        self.drop  = nn.Dropout(p=0.5)
        
        # dummy parameter for tracking device
        self.dummy = nn.Parameter(torch.empty(0))
        
    def loss(self,p,y):
        
        # check device for model:
        device = self.dummy.device
        
        # p : softmax(x)
        loss_fnc = nn.NLLLoss().to(device=device)
        loss = loss_fnc(torch.log(p),y)
        
        return loss
     
    def enable_dropout(self):
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.train()

        return
 
    def forward(self, x):
        
        x = e2nn.GeometricTensor(x, self.input_type)
        
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        
        x = self.gpool(x)
        x = x.tensor
        
        x = x.view(x.size()[0], -1)
        
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.drop(x)
        x = self.fc3(x)
    
        return x

In [7]:
from torchsummary import summary

model = DNSteerableLeNet(1, 1, 128+1, kernel_size=5, N=16).cuda()

summary(model, (1, 128+1, 128+1))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
SingleBlockBasisExpansion-1            [-1, 32, 1, 25]               0
BlocksBasisExpansion-2                [-1, 1, 25]               0
            R2Conv-3        [-1, 192, 127, 127]               0
              ReLU-4        [-1, 192, 127, 127]               0
PointwiseMaxPoolAntialiased-5          [-1, 192, 63, 63]               0
SingleBlockBasisExpansion-6           [-1, 32, 32, 25]               0
BlocksBasisExpansion-7              [-1, 192, 25]               0
            R2Conv-8          [-1, 512, 61, 61]               0
              ReLU-9          [-1, 512, 61, 61]               0
PointwiseMaxPoolAntialiased-10          [-1, 512, 30, 30]               0
     GroupPooling-11           [-1, 16, 30, 30]               0
           Linear-12                  [-1, 120]       1,728,120
           Linear-13                   [-1, 84]          10,164
  

In [19]:
class NerdyNet(nn.Module):
    def __init__(self, in_channels, out_channels, imsize, kernel_size=3, N=8):
        super(NerdyNet, self).__init__()

        self.r2_act = gspaces.FlipRot2dOnR2(N)

        in_type = e2nn.FieldType(self.r2_act, [self.r2_act.trivial_repr])
        self.input_type = in_type

        out_type = e2nn.FieldType(self.r2_act, 6*[self.r2_act.regular_repr])
        self.mask = e2nn.MaskModule(in_type, imsize, margin=1)

        self.conv1 = e2nn.R2Conv(in_type, out_type, kernel_size=3, padding=1, bias=False)
        self.relu1 = e2nn.ReLU(out_type, inplace=True)

        in_type = self.relu1.out_type
        out_type = e2nn.FieldType(self.r2_act, 16*[self.r2_act.regular_repr])
        
        self.conv2 = e2nn.R2Conv(in_type, out_type, kernel_size=3, padding=1, bias=False)
        self.relu2 = e2nn.ReLU(out_type, inplace=True)
        self.pool = e2nn.PointwiseMaxPoolAntialiased(out_type, kernel_size=2)
        
        in_type = self.pool.out_type
        out_type = e2nn.FieldType(self.r2_act, 36*[self.r2_act.regular_repr])
        
        self.conv3 = e2nn.R2Conv(in_type, out_type, kernel_size=3, padding=1, bias=False)
        self.relu3 = e2nn.ReLU(out_type, inplace=True)
        
        in_type = self.relu3.out_type
        out_type = e2nn.FieldType(self.r2_act, 60*[self.r2_act.regular_repr])
        
        self.conv4 = e2nn.R2Conv(in_type, out_type, kernel_size=3, padding=1, bias=False)
        self.relu4 = e2nn.ReLU(out_type, inplace=True)
        
        # Decoder
        
        in_type = self.relu4.out_type
        out_type = e2nn.FieldType(self.r2_act, 36*[self.r2_act.regular_repr])
        
        self.conv5 = e2nn.R2Conv(in_type, out_type, kernel_size=3, padding=1, bias=False)
        self.relu5 = e2nn.ReLU(out_type, inplace=True)
        
        in_type = self.relu5.out_type
        out_type = e2nn.FieldType(self.r2_act, 16*[self.r2_act.regular_repr])
        
        self.conv6 = e2nn.R2Conv(in_type, out_type, kernel_size=3, padding=1, bias=False)
        self.relu6 = e2nn.ReLU(out_type, inplace=True)
        
#         in_type = self.relu6.out_type
#         out_type = e2nn.FieldType(self.r2_act, 6*[self.r2_act.regular_repr])
        
#         self.conv5 = e2nn.R2Conv(in_type, out_type, kernel_size=3, padding=1, bias=False)
#         self.relu5 = e2nn.ReLU(out_type, inplace=True)
        in_type = self.relu6.out_type
        out_type = e2nn.FieldType(self.r2_act, 6*[self.r2_act.regular_repr])
        
        self.conv7 = e2nn.R2ConvTransposed(in_type, out_type, kernel_size=2, stride=2, padding=0)
        
        
#         nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
#         nn.ReLU(inplace=True),
#         nn.MaxPool2d(kernel_size=2, stride=2),
#         nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
#         nn.ReLU(inplace=True),
#         nn.Conv2d(128, 192, kernel_size=3, stride=1, padding=1),
#         nn.ReLU(inplace=True),

#         nn.Conv2d(192, 128, kernel_size=3, padding=1),
#         nn.ReLU(inplace=True),
#         nn.Conv2d(128, 64, kernel_size=3, padding=1),
#         nn.ReLU(inplace=True),
#         nn.ConvTranspose2d(64, out_channels, kernel_size=2, stride=2, output_padding=0)

        
#         self.conv1 = e2nn.R2Conv(in_type, out_type, kernel_size=5, padding=1, bias=False)
#         self.relu1 = e2nn.ReLU(out_type, inplace=True)
#         self.pool1 = e2nn.PointwiseMaxPoolAntialiased(out_type, kernel_size=2)

#         in_type = self.pool1.out_type
#         out_type = e2nn.FieldType(self.r2_act, 16*[self.r2_act.regular_repr])
#         self.conv2 = e2nn.R2Conv(in_type, out_type, kernel_size=5, padding=1, bias=False)
#         self.relu2 = e2nn.ReLU(out_type, inplace=True)
#         self.pool2 = e2nn.PointwiseMaxPoolAntialiased(out_type, kernel_size=2)
        
#         self.gpool = e2nn.GroupPooling(out_type)

#         self.fc1   = nn.Linear(16*z*z, 120)
#         self.fc2   = nn.Linear(120, 84)
#         self.fc3   = nn.Linear(84, out_chan)
        
#         self.drop  = nn.Dropout(p=0.5)
        
#         # dummy parameter for tracking device
#         self.dummy = nn.Parameter(torch.empty(0))
        
    def forward(self, x):
        
        x = e2nn.GeometricTensor(x, self.input_type)
        
        x = self.conv1(x)
        x = self.relu1(x)
        
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool(x)
        
        x = self.conv3(x)
        x = self.relu3(x)
        x = self.conv4(x)
        x = self.relu4(x)

        x = self.conv5(x)
        x = self.relu5(x)
        
        x = self.conv6(x)
        x = self.relu6(x)
        
        x = self.conv7(x)
        
        x = x.tensor
        
#         x = x.view(x.size()[0], -1)
        
#         x = F.relu(self.fc1(x))
#         x = F.relu(self.fc2(x))
#         x = self.drop(x)
#         x = self.fc3(x)
    
        return x

In [20]:
n = NerdyNet(1, 1, 128).cuda()

WARNING! The basis for the block expansion of the filter is empty!


In [38]:
class DNSteerableLeNet(nn.Module):
    def __init__(self, in_chan, out_chan, imsize, kernel_size=5, N=8):
        super(DNSteerableLeNet, self).__init__()
        
        z = 0.5*(imsize - 2)
        z = int(0.5*(z - 2))
        
        self.r2_act = gspaces.FlipRot2dOnR2(N)
        
        in_type = e2nn.FieldType(self.r2_act, [self.r2_act.trivial_repr])
        self.input_type = in_type
        
        out_type = e2nn.FieldType(self.r2_act, 6*[self.r2_act.regular_repr])
        self.mask = e2nn.MaskModule(in_type, imsize, margin=1)
        self.conv1 = e2nn.R2Conv(in_type, out_type, kernel_size=5, padding=1, bias=False)
        self.relu1 = e2nn.ReLU(out_type, inplace=True)
        self.pool1 = e2nn.PointwiseMaxPoolAntialiased(out_type, kernel_size=2)

        in_type = self.pool1.out_type
        out_type = e2nn.FieldType(self.r2_act, 16*[self.r2_act.regular_repr])
        self.conv2 = e2nn.R2Conv(in_type, out_type, kernel_size=5, padding=1, bias=False)
        self.relu2 = e2nn.ReLU(out_type, inplace=True)
#         self.pool2 = e2nn.PointwiseMaxPoolAntialiased(out_type, kernel_size=2)
        
#         self.gpool = e2nn.GroupPooling(out_type)

#         self.fc1   = nn.Linear(16*z*z, 120)
#         self.fc2   = nn.Linear(120, 84)
#         self.fc3   = nn.Linear(84, out_chan)
        
#         self.drop  = nn.Dropout(p=0.5)
        
        # dummy parameter for tracking device
#         self.dummy = nn.Parameter(torch.empty(0))
        
        in_type = self.rel2.out_type
        out_type = e2nn.FieldType(self.r2_act, 6*[self.r2_act.regular_repr])
        
        
    def loss(self,p,y):
        
        # check device for model:
        device = self.dummy.device
        
        # p : softmax(x)
        loss_fnc = nn.NLLLoss().to(device=device)
        loss = loss_fnc(torch.log(p),y)
        
        return loss
     
    def enable_dropout(self):
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.train()

        return
 
    def forward(self, x):
        
        x = e2nn.GeometricTensor(x, self.input_type)
        
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        
        x = self.conv2(x)
        x = self.relu2(x)
#         x = self.pool2(x)
        
#         x = self.gpool(x)
        x = x.tensor
        
        x = x.view(x.size()[0], -1)
        
#         x = F.relu(self.fc1(x))
#         x = F.relu(self.fc2(x))
#         x = self.drop(x)
#         x = self.fc3(x)
    
        return x

In [39]:
dns = DNSteerableLeNet(1, 1, 128).cuda()

In [40]:
summary(dns, (1, 128, 128))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
SingleBlockBasisExpansion-1            [-1, 16, 1, 25]               0
BlocksBasisExpansion-2                [-1, 1, 25]               0
            R2Conv-3         [-1, 96, 126, 126]               0
              ReLU-4         [-1, 96, 126, 126]               0
PointwiseMaxPoolAntialiased-5           [-1, 96, 63, 63]               0
SingleBlockBasisExpansion-6           [-1, 16, 16, 25]               0
BlocksBasisExpansion-7               [-1, 96, 25]               0
            R2Conv-8          [-1, 256, 61, 61]               0
              ReLU-9          [-1, 256, 61, 61]               0


AttributeError: 'int' object has no attribute 'numpy'